In [0]:
from pyspark.sql import SparkSession
from collections import defaultdict
from pyspark.sql import functions as F
from functools import reduce
import os
import shutil

import re

In [0]:
#Initialize Spark session
spark = SparkSession.builder.getOrCreate()

In [0]:
# mount point where the Azure Storage is mounted
mount = "/mnt/globalcsvdata"

# Common delimiters to try
delimiters = [",", ";", "\t", "|"]

# Dictionary to store DataFrames for each CSV
dfs = {}

In [0]:
# Loop over all subfolders in the  mount

for folder in dbutils.fs.ls(mount):
    if folder.isDir():  # Check if the item is a directory
        # Loop over all files inside the subfolder
        for file in dbutils.fs.ls(folder.path):
            if file.name.endswith(".csv"):  # Process only CSV files
                # Generate a valid key name for the dictionary
                # Replace all non-alphanumeric characters with underscores
                key_name = re.sub(r'\W+', '_', file.path.replace("/mnt/globalcsvdata", '').replace('.csv','').strip('_'))
                
                # Try reading with different delimiters until one works
                df = None
                for d in delimiters:
                    temp_df = spark.read.option("header", True).option("inferSchema", True).option("sep", d).csv(file.path)
                    
                    # Check if header was split correctly (more than 1 column)
                    if len(temp_df.columns) > 1:
                        df = temp_df
                        break
                
                # Store the DataFrame if successfully parsed
                if df is not None:
                    # Extract region from the DataFrame name ("africa", "america", "asia", "europe")
                    # Here I take the first token after 'dbfs_' in the name
                    parts = key_name.split('_')
                    region = parts[1] if len(parts) > 1 else "unknown"
                    
                    # Add region column
                    df = df.withColumn("region", F.lit(region))
                    
                    #store data frame in dictionnary
                    dfs[key_name] = df
                else:
                    print(f"Could not read : {file.path}")

# Print the total number of DataFrames created
print(str(len(dfs)) +" DataFrames created")

4 DataFrames created


In [0]:
# Loop each DataFrame in the dictionary
for name, df in dfs.items():
    print(f"DataFrame name: {name}")  # Print the generated name/key
    df.show(5)  # Show the first 5 rows of the DataFrame
    print("-" * 50)  # Separator for readability

DataFrame name: dbfs_africa_bu_africa
+-----------+---------------+--------+-------+---------+------+
|id_batterie|date_production|capacité|tension|fabricant|region|
+-----------+---------------+--------+-------+---------+------+
|      AF001|     2025-01-01|    85.5|    400|     Beta|africa|
|      AF002|     2025-01-02|    90.0|    405|     Beta|africa|
|      AF003|     2025-01-03|    82.3|    398|     Beta|africa|
|      AF004|     2025-01-04|    88.1|    402|     Beta|africa|
|      AF005|     2025-01-05|    87.0|    401|     Beta|africa|
+-----------+---------------+--------+-------+---------+------+
only showing top 5 rows

--------------------------------------------------
DataFrame name: dbfs_america_bu_america
+----------+----------+------------+-------+------------+-------+
|battery_id| prod_date|capacity_kWh|voltage|manufacturer| region|
+----------+----------+------------+-------+------------+-------+
|     AM001|2025-01-01|        85.5|  400.0|       Gamma|america|
|     

In [0]:
import unicodedata
from pyspark.sql import functions as F

# Function to normalize column names
def normalize_col(col_name):
    # Remove accents
    col = ''.join(c for c in unicodedata.normalize('NFKD', col_name) if not unicodedata.combining(c))
    # Lowercase and replace non-alphanumeric with underscore
    col = re.sub(r'\W+', '_', col.lower())
    return col.strip('_')

# Step 1: Normalize column names for all DataFrames
normalized_dfs = {}
for name, df in dfs.items():
    for old_col in df.columns:
        df = df.withColumnRenamed(old_col, normalize_col(old_col))
    normalized_dfs[name] = df

# Step 2: Build global schema from all normalized columns
all_columns = set()
for df in normalized_dfs.values():
    all_columns.update(df.columns)

all_columns.add("region")
all_columns = sorted(all_columns)

# Step 3: Standardize each DataFrame to the global schema
standardized_dfs = []
for name, df in normalized_dfs.items():
    # Add missing columns
    for col in all_columns:
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None))
    
    # Add region from DataFrame name
    region = name.split("_")[1] if "_" in name else "unknown"
    df = df.withColumn("region", F.lit(region))
    
    # Reorder columns
    df = df.select(all_columns)
    standardized_dfs.append(df)

# Step 4: Union all DataFrames
final_df = standardized_dfs[0]
for df in standardized_dfs[1:]:
    final_df = final_df.unionByName(df)

# Show result
print(f"Final DataFrame has {len(all_columns)} columns")
final_df.show(10, truncate=False)


Final DataFrame has 16 columns
+----------+--------+------------+---------------+--------+----+---------+----+-----------+------------+---------+---------------+------+-------+------+-------+
|battery_id|capacite|capacity_kwh|date_production|dateprod|fab |fabricant|id  |id_batterie|manufacturer|prod_date|production_date|region|tension|v_unit|voltage|
+----------+--------+------------+---------------+--------+----+---------+----+-----------+------------+---------+---------------+------+-------+------+-------+
|NULL      |85.5    |NULL        |2025-01-01     |NULL    |NULL|Beta     |NULL|AF001      |NULL        |NULL     |NULL           |africa|400    |NULL  |NULL   |
|NULL      |90.0    |NULL        |2025-01-02     |NULL    |NULL|Beta     |NULL|AF002      |NULL        |NULL     |NULL           |africa|405    |NULL  |NULL   |
|NULL      |82.3    |NULL        |2025-01-03     |NULL    |NULL|Beta     |NULL|AF003      |NULL        |NULL     |NULL           |africa|398    |NULL  |NULL   |
|NU

In [0]:
# ----------------------------------------------------
# ASSUMPTION:
# Each Business Unit (BU) uses a specific schema/format
# that does not change over time.
# Therefore, we can define a mapping to standardize 
# column names across all DataFrames.
# ----------------------------------------------------

# Define mapping rules for each region
# Keys = region name
# Values = dictionary mapping original column -> standardized column

column_mappings = {
    "africa": {
        "id_batterie": "battery_id",
        "date_production": "production_date",
        "capacité": "capacity_kWh",
        "tension": "voltage",
        "fabricant": "manufacturer"
    },
    "america": {
        "battery_id": "battery_id",
        "prod_date": "production_date",
        "capacity_kWh": "capacity_kWh",
        "voltage": "voltage",
        "manufacturer": "manufacturer"
    },
    "asia": {
        "ID": "battery_id",
        "DateProd": "production_date",
        "Capacité": "capacity_kWh",
        "Voltage": "voltage",
        "Fab": "manufacturer"
        # "V_unit" column is ignored since it's redundant ("V")
    },
    "europe": {
        "battery_id": "battery_id",
        "production_date": "production_date",
        "capacity_kWh": "capacity_kWh",
        "voltage": "voltage",
        "manufacturer": "manufacturer"
    }
}



In [0]:
# Apply mapping: loop on dfs,
# extract region from name, then rename columns
 
# Loop over all DataFrames in the dictionary
for name, df in dfs.items():
    # Extract region from DataFrame name
    parts = name.split('_')
    region = parts[1]
    
    if region in column_mappings:
        mapping = column_mappings[region]
        for old_col, new_col in mapping.items():
            if old_col in df.columns:
                df = df.withColumnRenamed(old_col, new_col)
        
        # Update DataFrame in dictionary
        dfs[name] = df

In [0]:
# Loop on DataFrame in the dictionary
for name, df in dfs.items():
    print(f"DataFrame name: {name}")  # Print the generated name/key
    df.show(5)  # Show the first 5 rows of the DataFrame
    print("-" * 50)  # Separator for readability

DataFrame name: dbfs_africa_bu_africa
+----------+---------------+------------+-------+------------+------+
|battery_id|production_date|capacity_kWh|voltage|manufacturer|region|
+----------+---------------+------------+-------+------------+------+
|     AF001|     2025-01-01|        85.5|    400|        Beta|africa|
|     AF002|     2025-01-02|        90.0|    405|        Beta|africa|
|     AF003|     2025-01-03|        82.3|    398|        Beta|africa|
|     AF004|     2025-01-04|        88.1|    402|        Beta|africa|
|     AF005|     2025-01-05|        87.0|    401|        Beta|africa|
+----------+---------------+------------+-------+------------+------+
only showing top 5 rows

--------------------------------------------------
DataFrame name: dbfs_america_bu_america
+----------+---------------+------------+-------+------------+-------+
|battery_id|production_date|capacity_kWh|voltage|manufacturer| region|
+----------+---------------+------------+-------+------------+-------+
|  

In [0]:
print(dfs[list(dfs.keys())[0]].columns)

['battery_id', 'production_date', 'capacity_kWh', 'voltage', 'manufacturer', 'region']


In [0]:
#Get the columns that are common to all DataFrames
common_columns = set(list(dfs.values())[0].columns)
for df in dfs.values():
    common_columns = common_columns.intersection(set(df.columns))
common_columns = list(common_columns)

print("Common columns:", common_columns)

Common columns: ['capacity_kWh', 'region', 'production_date', 'manufacturer', 'voltage', 'battery_id']


In [0]:
# Keep only these columns in each DataFrame
#List of dataframes
dfs_selected = [df.select(common_columns) for df in dfs.values()]

# Combine all DataFrames into one
df_global = reduce(lambda df1, df2: df1.unionByName(df2), dfs_selected)

In [0]:
# Check the result
df_global.show(100)
print("Total number of rows:", df_global.count())

+------------+-------+---------------+------------+-------+----------+
|capacity_kWh| region|production_date|manufacturer|voltage|battery_id|
+------------+-------+---------------+------------+-------+----------+
|        85.5| africa|     2025-01-01|        Beta|  400.0|     AF001|
|        90.0| africa|     2025-01-02|        Beta|  405.0|     AF002|
|        82.3| africa|     2025-01-03|        Beta|  398.0|     AF003|
|        88.1| africa|     2025-01-04|        Beta|  402.0|     AF004|
|        87.0| africa|     2025-01-05|        Beta|  401.0|     AF005|
|        89.2| africa|     2025-01-06|        Beta|  399.0|     AF006|
|        84.5| africa|     2025-01-07|        Beta|  403.0|     AF007|
|        91.0| africa|     2025-01-08|        Beta|  407.0|     AF008|
|        83.3| africa|     2025-01-09|        Beta|  396.0|     AF009|
|        86.5| africa|     2025-01-10|        Beta|  400.0|     AF010|
|        88.0| africa|     2025-01-11|        Beta|  402.0|     AF011|
|     

In [0]:
# Define the full path to save the CSV
output_path = "mnt/consolidatedcsvdata"

# Save dataframe as a single CSV file
(df_global.coalesce(1)
   .write
   .mode("overwrite")
   .option("header", "true")
   .csv(output_path))

print(f"Dataframe saved at {output_path}")


Dataframe saved at mnt/consolidatedcsvdata
